# YAMNET Transfer learning


## The typical transfer-learning workflow

This leads us to how a typical transfer learning workflow can be implemented in Keras:

* Instantiate a base model and load pre-trained weights into it.
* Freeze all layers in the base model by setting trainable = False.
* Create a new model on top of the output of one (or several) layers from the base model.
* Train your new model on your new dataset.

Note that an alternative, more lightweight workflow could also be:

* Instantiate a base model and load pre-trained weights into it.
* Run your new dataset through it and record the output of one (or several) layers from the base model. This is called feature extraction.
* Use that output as input data for a new, smaller model.
* A key advantage of that second workflow is that you only run the base model once on your data, rather than once per epoch of training. So it's a lot faster & cheaper.


References: 
https://www.tensorflow.org/tutorials/audio/transfer_learning_audio


In [1]:
import pandas as pd
import tensorflow as tf 
import h5py
import os
import tensorflow_hub as hub
import resampy
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import datetime
import glob
import resampy

# Model specific
import params as yamnet_params
import yamnet as yamnet_model
from IPython.display import Audio, display

print("running TF version",tf.__version__)

ModuleNotFoundError: ignored

In [ ]:
df_class_map = pd.read_csv("yamnet_class_map.csv")
df_custom_class = pd.read_csv("category_map.csv", sep=";")

mid_dict = {mid:i for i, mid in enumerate(df_class_map["mid"].values)}
index_dict = {i:mid for i, mid in enumerate(df_class_map["mid"].values)}

custom_dict = {mid:i for i, mid in enumerate(df_custom_class["mid"].values)}

def mid_to_index(mid):
    return mid_dict[mid]

def index_to_mid(mid):
    return index_dict[mid]

In [ ]:
# Clone model repo
#!git clone https://github.com/tensorflow/models.git
#%cd models/research/audioset/yamnet

# Download YAMNet data
#!curl -O https://storage.googleapis.com/audioset/yamnet.h5

In [ ]:
# Set DATASET ROOT FOLDERS

eval_folder = "/Volumes/AAC_Book/AUDIOSET_STRONG/eval_16k/"
train_folder = "/Volumes/AAC_Book/AUDIOSET_STRONG/train_16k/"

assert os.path.exists(eval_folder), "Carpeta evaluacion no existe"
assert os.path.exists(train_folder), "Carpeta entrenamiento no existe"

# 1. Instantiate a base model and load pre-trained weights into it

Load the YAMNET Base model and built a submodel that outputs the 1024 embeddings from the global average pooling 2d

In [ ]:
# The graph is designed for a sampling rate of 16 kHz, but higher rates should work too.
# We also generate scores at a 10 Hz frame rate.
sr = 16000
params = yamnet_params.Params(sample_rate=sr, patch_hop_seconds=1)
print("Sample rate =", params.sample_rate)


# Set up the YAMNet model.
class_names = yamnet_model.class_names('yamnet_class_map.csv')
yamnet = yamnet_model.yamnet_frames_model(params)
yamnet.load_weights('yamnet.h5')

In [ ]:
yamnet.summary()

In [ ]:
embedding_layer = tf.keras.Model(inputs=yamnet.input,
                                 outputs=yamnet.get_layer('global_average_pooling2d').output)

embedding_layer.trainable = False

## Read metada csv file

We have to convert the label mid encodings to integers and then to one hot to serve as labels in a NN

In [ ]:
N = 50
metadata_train = pd.read_csv(f"../data_stratified/train_stratified_{N}_class.csv")
metadata_eval = pd.read_csv(f"../data_stratified/eval_stratified_{N}_class.csv")
metadata_train

We have to construct the audio path based on the location of the audio files, define a loading function that reads and audio file, check if it's mono and 16k

In [ ]:
metadata_train["path"] = train_folder + metadata_train["clip_id"] + ".wav"
metadata_eval["path"] = eval_folder + metadata_eval["clip_id"] + ".wav"

In [ ]:
def load_audio(filename):
    x, fs = sf.read(filename)
    
    if len(x.shape) > 1:
        x = np.mean(x, axis=1)
        
    #if fs != 16000:
        #x = resampy.resample(x,sr_orig=fs,sr_new=16000)
    return x

## Feature extraction

Run your new dataset through it and record the output of one (or several) layers from the base model. This is called feature extraction

With the feature extractor model we have to build a new dataset that matches the shape of the embeddings with the label, remember that this feature extractor outputs a 1024 dimensional vector per second and we have a dataset of audios that have different duration. Then we have to repeat the label of audio according to the embedding shape. 

Note that we're saving the results in memory, this approach is valid for small to medium size datasets. For large datasets is needed to implement feature and training pipelines

In [ ]:
metadata_train["label_id"] = metadata_train.apply(lambda x: custom_dict[x["label"]], axis=1)

filenames = metadata_train["path"]
targets = metadata_train["label_id"]
n_classes = 66 # fixed to the selection
embedding_size = 1024

all_embeddings = np.empty([1,embedding_size])
all_labels = np.zeros(1)
for file, label in zip(filenames, targets):
    x = load_audio(file)
    embeddings = embedding_layer(x)
    all_embeddings = np.concatenate( [all_embeddings, embeddings] )
    all_labels = np.concatenate( [all_labels,np.repeat(label,embeddings.shape[0])] )
    #print(all_embeddings.shape, all_labels.shape)

In [ ]:

metadata_eval["label_id"] = metadata_eval.apply(lambda x: custom_dict[x["label"]], axis=1)

filenames = metadata_eval["path"]
targets = metadata_eval["label_id"]
n_classes = 66 # fixed to the selection
embedding_size = 1024

all_embeddings_eval = np.empty([1,embedding_size])
all_labels_eval = np.zeros(1)

for file, label in zip(filenames, targets):
    x = load_audio(file)
    embeddings = embedding_layer(x)
    all_embeddings_eval = np.concatenate( [all_embeddings_eval, embeddings] )
    all_labels_eval = np.concatenate( [all_labels_eval,np.repeat(label,embeddings.shape[0])] )
    #print(all_embeddings.shape, all_labels.shape)

In [ ]:
# Plot embeddings to see the groupings

In [ ]:
# As a sanity cheeck let's hear 5 random examples of this dataset
rand_sample = metadata_train.sample(5)

for i, clip in rand_sample.iterrows():
    x, fs = sf.read(clip["path"])
    scores, embeddings, spectrogram = yamnet(x)
    mean_scores = np.mean(scores,axis=0)
    print("prediction: ",class_names[np.argmax(mean_scores)])
    print(mean_scores.shape)
    print("ground_truth: ", class_names[mid_to_index(clip.label)])
    display(Audio(data=x,rate=fs))

## Build the custom model

Using the extacted featured from the base model we will add a 128 dense layer plus a N_classes softmaz layer at the end

In [ ]:
### Model #######
inputs = tf.keras.layers.Input(shape=(embedding_size), dtype=tf.float32, name='input_embedding')
dense = tf.keras.layers.Dense(128, name='Hidden_1',
                              activation="relu",
                               kernel_initializer='ones',
    kernel_regularizer=tf.keras.regularizers.L1(0.01),
                             )(inputs)
#dense = tf.keras.layers.Dense(64, name='Hidden_2', activation="relu")(dense)
outputs =  tf.keras.layers.Dense(n_classes, name="output", activation="softmax")(dense)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

##################

opt = tf.keras.optimizers.SGD(learning_rate=0.0001)
model.compile(optimizer=opt,
              loss='categorical_crossentropy',
             metrics=["accuracy"],)
model.summary()

In [ ]:
x_tf = tf.constant(all_embeddings, dtype=tf.float32)
y_tf = tf.keras.utils.to_categorical(all_labels, num_classes = n_classes)

x_val = tf.constant(all_embeddings_eval, dtype=tf.float32)
y_val = tf.keras.utils.to_categorical(all_labels_eval, num_classes = n_classes)

In [ ]:

start_date = datetime.datetime.strftime( datetime.datetime.now(), format="%Y%m%d_%H:%M:%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=f"./logs_stratified/{start_date}_{N}",
                                                     )
model.fit(x=x_tf,
         y=y_tf,
         epochs=50,
          callbacks=[tensorboard_callback],
          validation_data=(x_val, y_val),
         batch_size=64)

In [ ]:
!tensorboard --logdir="./logs"

In [ ]:
all_embeddings.shape

print(all_labels.shape)